# Super AI SS6 OCR Pipeline (Colab)
รองรับ Google Colab + Drive และ output ตามกติกา `id,votes`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys

def run(cmd):
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

# Install OpenAI SDK for OpenRouter & Dependencies
# typhoon_ocr is optional (backend-dependent), but pandas/dotenv/tqdm are core.
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 
     'openai', 'pandas', 'tqdm', 'python-dotenv', 'typhoon_ocr'])

In [ ]:
# Auto-detect dataset path to fix FileNotFoundError
import os

possible_roots = [
    '/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/data',  # Standard Colab
    '/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569',       # Alternative Colab
    './data/data',                                                     # Local unzip style 1
    './data',                                                          # Local unzip style 2
    '.'                                                                # Current dir
]

TEMPLATE_CSV = None
BASE_DIR = ''

for root in possible_roots:
    path = os.path.join(root, 'submission_template.csv')
    if os.path.exists(path):
        TEMPLATE_CSV = path
        BASE_DIR = root
        break

if TEMPLATE_CSV is None:
    # Fallback default if nothing found
    BASE_DIR = '/content/drive/MyDrive/super-ai-engineer-season-6-ocr-2569/data'
    TEMPLATE_CSV = os.path.join(BASE_DIR, 'submission_template.csv')
    print(f"Warning: Dataset not found. Defaulting to: {TEMPLATE_CSV}")

IMAGE_DIR = os.path.join(BASE_DIR, 'images')
SAMPLE_LABEL_DIR = os.path.join(BASE_DIR, 'sample_labels')

# Working paths
PIPELINE_SCRIPT = '/content/embedded_pipeline.py'
OUTPUT_CSV = '/content/submission.csv'
SAMPLE_GOLD_CSV = '/content/sample_gold.csv'

print(f"Template path: {TEMPLATE_CSV}")
print(f"Image dir:     {IMAGE_DIR}")
print(f"Label dir:     {SAMPLE_LABEL_DIR}")

In [ ]:
import base64
import json
import os
import re
import threading
import time
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from difflib import get_close_matches
from pathlib import Path
from typing import Protocol, Dict, List, Optional

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

# --- 1. CONFIGURATION ---
API_KEY = os.getenv("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise ValueError("Missing API Key. Please set the OPENROUTER_API_KEY environment variable.")

MAX_WORKERS = os.cpu_count() or 4
CACHE_DIR = Path("./cache_sota_v25")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OCR_CACHE_CSV = CACHE_DIR / "ocr_cache.csv"
LLM_CACHE_CSV = CACHE_DIR / "llm_cache.csv"

# --- 2. CORE CLASSES ---

class RateLimiter:
    """Sliding-window rate limiter supporting multiple simultaneous constraints."""
    def __init__(self, limits: list[tuple[int, float]]):
        self._limits = limits
        self._windows: list[deque] = [deque() for _ in limits]
        self._lock = threading.Lock()

    def acquire(self) -> None:
        while True:
            with self._lock:
                now = time.monotonic()
                for (max_req, window), dq in zip(self._limits, self._windows):
                    while dq and now - dq[0] >= window:
                        dq.popleft()
                if all(len(dq) < max_req for (max_req, _), dq in zip(self._limits, self._windows)):
                    for dq in self._windows:
                        dq.append(now)
                    return
                wait = min((dq[0] + window - now) for (max_req, window), dq in zip(self._limits, self._windows) if len(dq) >= max_req)
                time.sleep(max(wait, 0.05))

class OcrBackend(Protocol):
    def __call__(self, image_path: Path) -> str: ...

_VLM_OCR_PROMPT = (
    "This is a page from a Thai official election result form (สส.6/1). "
    "Your task is to perform a complete, verbatim transcription of every piece of information on this page. "
    "Output only Markdown — no commentary, no explanations, nothing outside the transcription.\n\n"
    "Follow these rules strictly:\n"
    "1. **Transcribe every row** of every table without exception. Never skip, truncate, or summarise rows.\n"
    "2. **Preserve exact Thai text** for all party names (ชื่อพรรค), candidate names (ชื่อ-สกุล), "
    "   section labels, and header fields exactly as printed.\n"
    "3. **Convert Thai numerals** (๐๑๒๓๔๕๖๗๘๙) to Arabic digits (0–9) in all numeric fields.\n"
    "4. **For vote-count tables**, reproduce each data row as a Markdown table row with columns:\n"
    "   | หมายเลข | ชื่อ-สกุล | สังกัดพรรคการเมือง | คะแนน |\n"
    "   Include the header row and a separator row.\n"
    "5. **For summary statistics** (numbered items such as 1.1, 1.2, 2.1, 3.1 …), transcribe each item "
    "   on its own line in the format: `<item number> <label> <value>`.\n"
    "6. **Transcribe all header fields** (province, constituency/district name, polling station, date, "
    "   official signatures section labels) verbatim.\n"
    "7. If a cell is blank or illegible, write `-`.\n"
    "8. Do **not** add any text, commentary, or formatting that is not present in the image."
)

ocr_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)

def _ocr_openrouter_vlm(image_path: Path, model: str) -> str:
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = ocr_client.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": _VLM_OCR_PROMPT},
            ],
        }],
        temperature=0,
    )
    return resp.choices[0].message.content or ""

# --- BACKEND REGISTRY ---
OCR_BACKENDS: dict[str, OcrBackend] = {
    "gemini-2.0-flash": lambda p: _ocr_openrouter_vlm(p, model="google/gemini-2.0-flash-001"),
    "gemini-2.5-flash": lambda p: _ocr_openrouter_vlm(p, model="gemini-2.5-flash"),
}

OCR_BACKEND_NAME = "gemini-2.5-flash"

OCR_RATE_LIMITS = {
    "gemini-2.0-flash": [(10, 1.0), (50, 60.0)],
    "gemini-2.5-flash": [(15, 1.0), (75, 60.0)],
}

ocr_backend = OCR_BACKENDS[OCR_BACKEND_NAME]
ocr_rate_limiter = RateLimiter(limits=OCR_RATE_LIMITS.get(OCR_BACKEND_NAME, [(5, 1.0)]))

print(f"Setup complete. OCR backend: {OCR_BACKEND_NAME!r}")

In [ ]:
# Build sample gold labels from sample_labels for sanity scoring
import json
from pathlib import Path
import pandas as pd

template_df = pd.read_csv(TEMPLATE_CSV, usecols=['id', 'doc_id', 'row_num'])
template_df['row_num'] = template_df['row_num'].astype(int)

gold_rows = []
for p in sorted(Path(SAMPLE_LABEL_DIR).glob('*.json')):
    data = json.loads(p.read_text(encoding='utf-8'))
    doc_id = p.stem
    for r in data.get('results', []):
        try:
            row_num = int(r.get('number'))
            votes = ''.join(ch for ch in str(r.get('votes', '0')) if ch.isdigit()) or '0'
            gold_rows.append({'doc_id': doc_id, 'row_num': row_num, 'votes': votes})
        except Exception:
            pass

gold_df = pd.DataFrame(gold_rows)
sample_gold = template_df.merge(gold_df, on=['doc_id', 'row_num'], how='inner')
sample_gold = sample_gold[['id', 'votes']].copy()
sample_gold.to_csv(SAMPLE_GOLD_CSV, index=False)

print('sample label docs:', gold_df['doc_id'].nunique())
print('sample gold rows:', len(sample_gold))
print('saved:', SAMPLE_GOLD_CSV)
display(sample_gold.head())

In [ ]:
# --- 4. BATCH OCR (Matching solution.ipynb) ---

_ocr_write_lock = threading.Lock()

def _load_ocr_cache() -> dict[tuple[str, int], str]:
    if OCR_CACHE_CSV.exists():
        df = pd.read_csv(OCR_CACHE_CSV, dtype=str)
        # Ensure page is int
        df["page"] = pd.to_numeric(df["page"], errors='coerce').fillna(1).astype(int)
        df["ocr_text"] = df["ocr_text"].fillna("")
        return {(row.doc_id, int(row.page)): row.ocr_text for row in df.itertuples()}
    return {}

def _save_ocr_cache(cache: dict[tuple[str, int], str]) -> None:
    rows = [
        {"doc_id": doc_id, "page": page, "ocr_text": text}
        for (doc_id, page), text in cache.items()
    ]
    df = pd.DataFrame(rows, columns=["doc_id", "page", "ocr_text"])
    df.sort_values(["doc_id", "page"]).reset_index(drop=True).to_csv(OCR_CACHE_CSV, index=False)

def _ocr_one_page(doc_id: str, page_num: int, page: Path) -> tuple[str, int, Optional[str]]:
    with _ocr_write_lock:
        cached = ocr_cache.get((doc_id, page_num))
        if cached is not None and cached.strip():
            return doc_id, page_num, None  # already good

    ocr_rate_limiter.acquire()
    for attempt in range(3):
        try:
            # Use the global ocr_backend function
            text = ocr_backend(page)
            if text and text.strip():
                return doc_id, page_num, text
            print(f"  OCR returned blank [{doc_id} p{page_num}] attempt {attempt + 1}")
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  OCR attempt {attempt + 1} failed [{doc_id} p{page_num}]: {e}")
            time.sleep(2 ** attempt)
    return doc_id, page_num, ""

# Load Cache
ocr_cache = _load_ocr_cache()
print(f"OCR cache loaded: {len(ocr_cache)} pages.")

work: list[tuple[str, int, Path]] = []
for doc_id, pages in doc_pages.items():
    for page_num, page in enumerate(pages, start=1):
        key = (doc_id, page_num)
        if key not in ocr_cache or not ocr_cache[key].strip():
            work.append((doc_id, page_num, page))

print(f"Pages to OCR: {len(work)}")

if work:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(_ocr_one_page, doc_id, page_num, page): (doc_id, page_num)
            for doc_id, page_num, page in work
        }
        with tqdm(total=len(work), desc=f"Batch OCR [{OCR_BACKEND_NAME}]") as pbar:
            for future in as_completed(futures):
                d_id, pg_num, text = future.result()
                if text is not None:
                    with _ocr_write_lock:
                        ocr_cache[(d_id, pg_num)] = text
                        # Periodically save, or save strictly every time
                        _save_ocr_cache(ocr_cache)
                pbar.update(1)

print("OCR Loop Finished.")

# Prepare doc_ocr dict for next step
doc_ocr: dict[str, str] = {
    doc_id: "\n\n---PAGE BREAK---\n\n".join(
        ocr_cache.get((doc_id, page_num), "")
        for page_num, _ in enumerate(pages, start=1)
    )
    for doc_id, pages in doc_pages.items()
}

In [ ]:
# Optional: score on sample labels (lower is better)
import pandas as pd

pred = pd.read_csv(OUTPUT_CSV, dtype={'id': str, 'votes': str})
gold = pd.read_csv(SAMPLE_GOLD_CSV, dtype={'id': str, 'votes': str})
eval_df = gold.merge(pred, on='id', how='left', suffixes=('_gold', '_pred'))
eval_df['votes_pred'] = eval_df['votes_pred'].fillna('0')

def levenshtein(a: str, b: str) -> int:
    a = str(a)
    b = str(b)
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + cost)
            prev = cur
    return dp[m]

eval_df['lev'] = [levenshtein(g, p) for g, p in zip(eval_df['votes_gold'], eval_df['votes_pred'])]
print('sample rows:', len(eval_df))
print('sample mean levenshtein:', eval_df['lev'].mean())
display(eval_df.head())

In [ ]:
import pandas as pd
sub = pd.read_csv(OUTPUT_CSV)
display(sub.head())
print('rows:', len(sub), '| columns:', list(sub.columns))
assert list(sub.columns) == ['id', 'votes']
assert sub['votes'].astype(str).str.fullmatch(r'\d+').all(), 'votes ต้องเป็นเลขอารบิกล้วน'
print('submission ready:', OUTPUT_CSV)